# 3.1 经典推荐算法总结与横向对比

> 阅读版与 Web 应用内容一致；实验数值来自本 Notebook 的已执行输出。

## Goal

在完成四个独立实验后，从任务、表示、训练目标、指标、推理方式和工业边界六个角度进行横向比较。本文不重复完整实现，只提供章节地图、结果快照和选型评论。

## Setup

默认 `smoke` 档使用仓库内固定版本的 GroupLens **MovieLens latest-small** 真实行为子集，CPU 可重复执行；`full` 档只扩大真实数据规模与训练配置，不切换到合成数据。数据包含真实匿名用户、电影、评分和时间戳；实验只做确定性截取与任务重构，不随机制造交互、标签或行为序列。原始许可与引用保存在 `data/ml-latest-small/README.txt`。

**主要资料：** [MovieLens](https://grouplens.org/datasets/movielens/) · [MF](https://datajobs.com/data-science-repo/Recommender-Systems-[Netflix].pdf) · [FM](https://www.csie.ntu.edu.tw/~b97053/paper/Rendle2010FM.pdf) · [GBDT+LR](https://research.facebook.com/publications/practical-lessons-from-predicting-clicks-on-ads-at-facebook/)

In [1]:
from pathlib import Path
import os, sys, json
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.environ.setdefault("RECSYS_PROFILE", "smoke")
PROFILE = os.environ["RECSYS_PROFILE"]
from recsys_lab.data import load_movielens, movielens_provenance
real_ratings, real_movies = load_movielens()
REAL_DATASET = movielens_provenance(real_ratings)
print({"profile": PROFILE, "root": str(ROOT), "real_dataset": REAL_DATASET})
assert REAL_DATASET["randomly_fabricated_rows"] == 0

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'profile': 'smoke', 'root': '/workspace', 'real_dataset': {'dataset': 'MovieLens latest-small (GroupLens, generated 2018-09-26)', 'source': 'https://files.grouplens.org/datasets/movielens/ml-latest-small.zip', 'license_file': '/workspace/data/ml-latest-small/README.txt', 'rows_used': 26732, 'users_used': 120, 'items_used': 600, 'time_min_utc': '1996-10-17T11:51:49+00:00', 'time_max_utc': '2018-09-13T21:38:16+00:00', 'positive_rule': 'like := observed rating >= 4.0; very_like := observed rating >= 4.5', 'randomly_fabricated_rows': 0}}


## 学习路线

四种算法其实对应三种不同的“信息压缩”方式：

| 方法 | 压缩对象 | 学到的核心信息 | 常见位置 |
|---|---|---|---|
| UserCF / ItemCF | 共现邻域 | 谁和谁相似 | 召回、相关推荐 |
| MF | user–item 矩阵 | 用户与物品的低维隐语义 | 召回、评分预测 |
| FM | 稀疏特征交叉 | 任意两个特征的低秩交互 | CTR 排序 |
| GBDT+LR | 表格特征与树规则 | 非线性分桶、条件组合与概率校准 | CTR 排序 |

建议按顺序运行。前两类使用 user–item 行为；后两类使用“曝光—点击”样本。二者的数据生成机制和评价指标不同，不能把 RMSE、Recall 与 AUC 混为一谈。

## 来源论文与本章解读

本章不是按发表年份罗列名词，而是沿着“表示什么、怎样推理”阅读四条原始工作：

- **Resnick et al. (1994), GroupLens**：早期系统展示了如何依据用户间评分相似度形成邻域预测；关键遗产是“从群体行为借信号”，而不是某个固定相似度公式。
- **Sarwar et al. (2001), Item-based CF**：把邻域移到更稳定、可离线物化的 item 侧，直接影响了后来的相关推荐和倒排召回。
- **Koren, Bell & Volinsky (2009), Matrix Factorization Techniques**：以全局/用户/物品偏置加低秩内积解释评分；其 user/item 向量结构也是双塔召回的最小原型。
- **Rendle (2010), Factorization Machines**：用特征隐向量共享稀疏二阶交叉统计，并用恒等式把计算从 $O(n^2k)$ 降为 $O(nk)$。
- **He et al. (2014), Practical Lessons from Predicting Clicks on Ads at Facebook**：树负责产生非线性叶规则，LR 负责稀疏组合与概率输出，代表经典的两阶段 CTR 工程路线。

阅读时请区分两组问题：CF/MF 面向 user–item 评分或 Top-K；FM/GBDT+LR 面向曝光后的点击概率。不同任务的指标不能直接排名。

## Steps

## 1. 四个独立实验

| Notebook | 核心问题 | 主要指标 |
|---|---|---|
| 3.1.1 协同过滤 | 谁与谁相似，如何用邻域产生候选？ | HR@K、Recall@K、Coverage |
| 3.1.2 矩阵分解（BiasMF） | 如何把稀疏矩阵压缩成用户/物品隐向量？ | RMSE、HR@K |
| 3.1.3 FM | 如何在线性复杂度内学习稀疏二阶交互？ | AUC、LogLoss |
| 3.1.4 GBDT+LR | 如何用树叶规则生成特征，再由 LR 校准概率？ | AUC、LogLoss |

这四篇均可独立从头执行，不依赖其他 Notebook 的内存状态。

## 2. 结果快照

以下数值由四个算法 Notebook 在执行末尾写入 `results/chapter_3_1/*.json`，本总结只读取产物，不手填实验值。CF/MF 与 FM/GBDT+LR 属于不同任务，表格用于确认代码路径和理解指标，不能按数值大小直接排名。

In [2]:
import json
import pandas as pd

result_dir = ROOT / "results" / "chapter_3_1"
result_files = sorted(result_dir.glob("*.json"))
assert len(result_files) == 4, f"需要先执行四个算法 Notebook；当前产物：{[p.name for p in result_files]}"
records = []
for path in result_files:
    payload = json.loads(path.read_text(encoding="utf-8"))
    records.extend(payload["records"])
comparison = pd.DataFrame(records)
display(comparison.round(4))
print("指标来源：", [p.name for p in result_files])

,algorithm,task,metric,value,secondary_metric,secondary_value,online_inference,source_notebook
0,UserCF,Top-K,HR@10 ↑,0.1250,Coverage,0.3222,聚合相似用户的历史,3_1_1_collaborative_filtering
1,ItemCF,Top-K,HR@10 ↑,0.1250,Coverage,0.3278,聚合历史物品的近邻,3_1_1_collaborative_filtering
2,FM,CTR,AUC ↑,0.5964,LogLoss ↓,2.6061,线性项 + 二阶隐向量交互,3_1_3_factorization_machine
3,GBDT+LR,CTR,AUC ↑,0.6376,LogLoss ↓,0.7414,树叶编码 → LR 概率,3_1_4_gbdt_lr
4,BiasMF,评分预测,RMSE ↓,1.1905,HR@10,0.2083,user/item 向量内积,3_1_2_matrix_factorization


指标来源： ['collaborative_filtering.json', 'factorization_machine.json', 'gbdt_lr.json', 'matrix_factorization.json']


## 3. 横向评论

| 维度 | CF | MF | FM | GBDT+LR |
|---|---|---|---|---|
| 输入 | user–item 行为 | user–item 评分/行为 | 任意稀疏与连续特征 | 表格特征 |
| 表示 | 邻域/相似度 | 低维 embedding | 特征 embedding | 树叶规则 |
| 交互能力 | 共现传播 | user-item 内积 | 全部二阶交互 | 非线性条件组合 |
| 冷启动 | 弱 | 弱 | 可加入内容特征 | 可加入内容特征 |
| 序列能力 | 无 | 无 | 无 | 无 |
| 典型位置 | 召回/相关推荐 | 召回/评分 | CTR 排序 | CTR 排序 |
| 主要风险 | 稀疏、热门偏差 | 未观察样本、内积限制 | 仅二阶 | 两阶段失配、规则漂移 |

技术演进不是简单替换关系：ItemCF 常作为覆盖与兜底通道长期保留；MF 是双塔向量召回的最小原型；FM 和 GBDT+LR 则代表排序阶段两条经典特征交叉路线。

## Checks

- 比较 Top-K 模型时同时看命中、覆盖和新颖度。
- 比较 CTR 模型时至少同时看 AUC、LogLoss 与校准。
- 只在相同数据、相同时间切分、相同负样本下横向比较。
- smoke 指标用于代码与理解验证，不当作公开 benchmark。

In [3]:
assert len(comparison) == 5
assert set(comparison.task) == {'Top-K', '评分预测', 'CTR'}
assert comparison.source_notebook.nunique() == 4
print('PASS：章节总结从四个独立实验聚合了全部经典算法与三类任务。')

PASS：章节总结从四个独立实验聚合了全部经典算法与三类任务。


## Next Steps

按顺序阅读四篇算法 Notebook。若目标是构建系统基线，建议先落地 ItemCF 与 GBDT+LR/FＭ，再根据目录规模和特征类型引入 MF/双塔与 DeepFM。